In [1]:
from transitions import Machine
from typing import List
class State:
    machine: Machine
    lock: bool = True

    def is_locked(self) -> bool:
        return self.lock
    def is_unlocked(self) -> bool:
        return not self.lock
    
state = State()
states = [{'name':'idle',}, 
            {'name':'active',}]

state.machine = Machine(model=state, states=states, initial='idle')
state.machine.add_transition('change', 'idle', 'active', conditions=['is_unlocked'])
state.machine.add_transition('change', 'active', 'idle', conditions=['is_unlocked'])

state.change()

print(state.state)

idle


In [2]:
class Entity(State):
    sub_state: State

    def __init__(self):
        states = [{'name':'locked', 'on_enter':['update_lock']}, 
                    {'name':'unlocked', 'on_enter':['update_lock']}]
        transitions = [
            {'trigger':'lock', 'source':'unlocked', 'dest':'locked'},
            {'trigger':'unlock', 'source':'locked', 'dest':'unlocked'},
        ]
        machine = Machine(model=self, states=states, transitions=transitions, initial='locked')
        self.machine = machine

        state = State()
        states = [{'name':'idle',}, 
                    {'name':'active',}]

        state.machine = Machine(model=state, states=states, initial='idle')
        state.machine.add_transition('change', 'idle', 'active', conditions=['is_unlocked'])
        state.machine.add_transition('change', 'active', 'idle', conditions=['is_unlocked'])
        self.sub_state = state

    def update_lock(self):
        self.sub_state.lock = self.state == 'locked'

entity = Entity()
entity.sub_state.change()
print(entity.sub_state.state)
entity.unlock()
entity.sub_state.change()
print(entity.sub_state.state)

idle
active


In [3]:
class Entity(State):
    sub_state: State

    def __init__(self):
        states = [{'name':'locked', 'on_enter':['update_lock']}, 
                    {'name':'unlocked', 'on_enter':['update_lock']}]
        transitions = [
            {'trigger':'lock', 'source':'unlocked', 'dest':'locked'},
            {'trigger':'unlock', 'source':'locked', 'dest':'unlocked'},
        ]
        machine = Machine(model=self, states=states, transitions=transitions, initial='locked')
        self.machine = machine

        state = State()
        states = [{'name':'idle',}, 
                    {'name':'active',}]

        state.machine = Machine(model=state, states=states, initial='idle')
        state.machine.add_transition('change', 'idle', 'active', conditions=['is_unlocked'])
        state.machine.add_transition('change', 'active', 'idle', conditions=['is_unlocked'])
        self.sub_state1 = state

        state = State()
        states = [{'name':'stateA',}, 
                    {'name':'stateB',}]

        state.machine = Machine(model=state, states=states, initial='stateA')
        state.machine.add_transition('change', 'stateA', 'stateB', conditions=['is_unlocked'])
        state.machine.add_transition('change', 'stateB', 'stateA', conditions=['is_unlocked'])
        self.sub_state2 = state

        self.substates = [self.sub_state1, self.sub_state2]

    def update_lock(self):
        for sub_state in self.substates:
            sub_state.lock = self.state == 'locked'
            sub_state.change()

entity = Entity()
print(entity.sub_state1.state, entity.sub_state2.state)
entity.unlock()
print(entity.sub_state1.state, entity.sub_state2.state)

idle stateA
active stateB


In [5]:
from type_protocols import StatefulObject, StateStoreObject
from core_components.entities.types import GameEntity
from core_components.maps.tiles.base import TileCoordinate, TileTuple
from typing import Any, Dict, Protocol, Tuple, TypedDict, runtime_checkable
import time
import numpy as np

@runtime_checkable
class EntitySubState(Protocol):
    machine: Machine
    name: str
    store: StatefulObject
    state_bit_dtypes: Tuple[np.dtype, ...]

@runtime_checkable
class EntityParentState(Protocol):
    machine: Machine
    substates: list[EntitySubState]


class BaseSubState:
    machine: Machine
    name: str
    store: StatefulObject

    # Set defaults in subclasses
    _state_bits: Tuple[str, ...]
    _states: Tuple[Dict, ...]
    _transitions: Tuple[Dict, ...]
    _initial_state: str

    def __init__(self, name: str, store: StatefulObject):
        self.name = name
        self.store = store
    
        machine = Machine(model=self, states=self._states, transitions=self._transitions, initial=self._initial_state)
        self.machine = machine

    def set_bits(self) -> None:
        pass # Define in subclasses


class BaseParentState:
    machine: Machine
    substates: list[BaseSubState]
    state_vector: Dict[str, bool]
    state_vector_dtype: np.dtype
    # Define in subclasses
    _substates_manifest: Tuple[Tuple[str, type[BaseSubState]], ...]

    def __init__(self):
        self.substates = []
        self.state_vector = {}
    
        for name, substate in self._substates_manifest:        
            self._add_substate(substate(name=name, store=self))

    def update(self) -> None:
        for substate in self.substates:
            substate.set_bits()
            substate.update() # type: ignore
    
    def _add_substate(self, substate: BaseSubState) -> None:
        self.substates.append(substate)
        for bit in substate._state_bits:
            self.state_vector[bit] = False
        self.__setattr__(substate.name.lower(), substate) #type: ignore



In [6]:
class TestSubState(BaseSubState):
    _state_bits = ('on_map',)
    _states = ({'name': 'on_map'}, {'name': 'off_map'})
    _transitions = (
        {'trigger':'update', 'source':'off_map', 'dest':'on_map'},
        {'trigger':'update', 'source':'on_map', 'dest':'off_map'},
    )
    _initial_state = 'off_map'
    
    def set_bits(self) -> None:
        self.store.state_vector['on_map'] = True  # type: ignore
        
class TestState(BaseParentState):
    _substates_manifest = (
        ("Spawn", TestSubState),
    )

a = TestState()
print(a.state_vector.values(), a.spawn.state)
a.update()
print(a.state_vector.values(), a.spawn.state)

dict_values([False]) off_map
dict_values([True]) on_map


In [7]:
class GameStore:
    machine: Machine
    pass

class SpawnSubState(BaseSubState):
    """
    A generic substate for game entities.

    Duck Types: EntitySubState
    """
    _state_bits = ('on_map',)
    _states = ({'name': 'in_play'}, {'name': 'not_in_play'})
    _transitions = (
        {'trigger':'update', 'source':'not_in_play', 'dest':'in_play', 'conditions':['is_on_map']},
        {'trigger':'update', 'source':'in_play', 'dest':'not_in_play', 'conditions':['is_not_on_map']},
    )
    _initial_state = 'not_in_play'

    def set_bits(self) -> None:
        self.store.state_vector['on_map'] = self.store.location is not None #type: ignore

    # All substates must have the primary state bit methods
    def is_on_map(self) -> bool:
        return self.store.state_vector['on_map'] # type: ignore

    def is_not_on_map(self) -> bool:
        return not self.store.state_vector['on_map'] # type: ignore
    
    
class BaseGameEntity(BaseParentState):
    """
    A generic object to represent players, enemies, items, etc.

    Duck Types: StatefulObject, StateStoreObject, GameEntity
    """
    store: StatefulObject | None
    machine: Machine
    name: str
    symbol: str
    color: Tuple[int, int, int] # Do this like the maps. Numpy datatypes mapped to state.
    location: TileCoordinate | None

    # Substate definition
    _substates_manifest = (
        ("spawn", SpawnSubState),
    )

    def __init__(self,
                 store: GameStore,
                 *,                 
                 location: Tuple[int, int] | TileCoordinate | None = None,
                 name: str="<Unnamed>", 
                 symbol: str=' ', 
                 color: Tuple[int, int, int]=(0,0,0)) -> None:
        super().__init__()
        self.store = store
        parent_map_size = TileTuple(([100], [100]))
        
        if isinstance(location, tuple):
            self.location = TileCoordinate.from_tuple(location, parent_map_size=parent_map_size)
        else:
            self.location = location

        self.symbol = symbol
        self.color = color
        self.name = name
        self.update()

In [8]:
class DummyParent:
    def __init__(self):
        self.state_vector = np.array([(False,)], dtype=[('on_map', 'bool')])

a = SpawnSubState(name="Spawn", store=DummyParent())  # type: ignore
a.update()
print(a.state)
a.store.state_vector['on_map'] = True
a.update()
print(a.state)
a.update()
print(a.state)
a.store.state_vector['on_map'] = False
a.update()
print(a.state)

a = BaseGameEntity(store=GameStore(), name="Test Entity")  # type: ignore
print(a.state_vector, a.spawn.state)
a.state_vector['on_map'] = True
a.update()
print(a.spawn.state)

not_in_play
in_play
in_play
not_in_play
{'on_map': False} not_in_play
not_in_play


In [9]:
from type_protocols import StatefulObject


class TargetedSubState(SpawnSubState):
    _state_bits = ('has_targeter',)
    _states = ({'name':'targeted', 'on_enter': ['update']}, 
                 {'name':'not_targeted', 'on_enter': ['update']},
                 {'name':'unknown', 'on_enter': ['update']},)
    _transitions = (
            {'trigger':'update', 'source':['not_targeted', 'unknown'], 'dest':'targeted', 'conditions':['is_on_map', 'is_targeted']},
            {'trigger':'update', 'source':['targeted', 'unknown'], 'dest':'not_targeted', 'conditions':['is_on_map', 'is_not_targeted']},
            {'trigger':'update', 'source':['targeted', 'not_targeted'], 'dest':'unknown', 'conditions':['is_not_on_map']},
        )
    _initial_state = 'not_targeted'
    
    def set_bits(self) -> None:
        self.store.state_vector['has_targeter'] = self.store.targeter is not None  # type: ignore

    def is_targeted(self) -> bool:
        return self.store.state_vector['has_targeter']  # type: ignore

    def is_not_targeted(self) -> bool:
        return not self.store.state_vector['has_targeter']  # type: ignore

    
class BaseTargetableEntity(BaseGameEntity):
    targeter: StatefulObject | None = None
    targeted: TargetedSubState
    _substates_manifest = (
        ("spawn", SpawnSubState),
        ("targeted", TargetedSubState),
    )
    

a = BaseTargetableEntity(store=GameStore(), location=(10,15), name="Targetable Entity", symbol='T', color=(0,255,0))
print(a.state_vector, a.spawn.state, a.targeted.state, a.location)
a.targeter = 10
a.update()
print(a.state_vector, a.spawn.state, a.targeted.state, a.location)

{'on_map': True, 'has_targeter': False} in_play not_targeted TileCoordinate(x=10, y=15, parent_map_size=([100], [100]))
{'on_map': True, 'has_targeter': True} in_play targeted TileCoordinate(x=10, y=15, parent_map_size=([100], [100]))


In [10]:
a = BaseTargetableEntity(store=GameStore())

a.targeter = 10
a.update()
print(a.targeted.state, a.state_vector)

a.targeter = None
a.update()
print(a.targeted.state, a.state_vector)
a.location = TileCoordinate.from_tuple((5,5), parent_map_size=TileTuple(([100],[100])))
a.update()
print(a.targeted.state, a.state_vector)

a.targeter = 20
a.update()
print(a.targeted.state, a.state_vector)

a.targeter = None
a.update()
print(a.targeted.state, a.state_vector)

unknown {'on_map': False, 'has_targeter': True}
unknown {'on_map': False, 'has_targeter': False}
not_targeted {'on_map': True, 'has_targeter': False}
targeted {'on_map': True, 'has_targeter': True}
not_targeted {'on_map': True, 'has_targeter': False}


In [11]:
class TargetingSubState(SpawnSubState):
    threat_level_threshold: int = 40

    _state_bits = ('target_in_fov', 'target_is_hostile')
    _states = ({'name':'stopped', 'on_enter':['update']},
                {'name':'searching', 'on_enter':['update']}, 
                {'name':'tracking', 'on_enter':['update']},
                {'name':'targeting', 'on_enter':['update']},
                {'name':'unknown', 'on_enter':['update']},)
    _transitions = (
            {'trigger':'update', 'source':['unknown', 'tracking', 'targeting'], 'dest':'searching', 'conditions':['is_on_map', 'has_no_hostile_target','has_no_visible_target']},
            {'trigger':'update', 'source':['searching', 'targeting'], 'dest':'tracking', 'conditions':['is_on_map', 'has_no_hostile_target','has_visible_target']},
            {'trigger':'update', 'source':['searching', 'tracking'], 'dest':'targeting', 'conditions':['is_on_map', 'has_visible_target', 'has_hostile_target']},
            {'trigger':'update', 'source':['searching', 'tracking', 'targeting'], 'dest':'unknown', 'conditions':['is_not_on_map']},
            {'trigger':'stop', 'source':['searching', 'tracking', 'targeting', 'stopped'], 'dest':'stopped'},
            {'trigger':'start', 'source':'stopped', 'dest':'searching', 'unless':'is_not_on_map'},
            )
    _initial_state = 'searching'

    def set_bits(self) -> None:
        self.store.state_vector['target_in_fov'] = self.target_in_fov()  # type: ignore
        self.store.state_vector['target_is_hostile'] = self.store.threat_level > self.threat_level_threshold and self.store.visible  # type: ignore
        #self.store.state_vector['entity_is_on_guard'] = self.store.target is not None  # type: ignore
    
    def target_in_fov(self) -> bool:
        return self.store.visible # Functions to be expanded later.
    
    def has_visible_target(self) -> bool:
        return self.store.state_vector['target_in_fov']  # type: ignore
    
    def has_no_visible_target(self) -> bool:
        return not self.store.state_vector['target_in_fov']  # type: ignore
    
    def has_hostile_target(self) -> bool:
        return self.store.state_vector['target_is_hostile']  # type: ignore
    
    def has_no_hostile_target(self) -> bool:
        return not self.store.state_vector['target_is_hostile']  # type: ignore
    
    def set_target(self, *, target: GameEntity) -> None:
        self.store.target = target  # type: ignore


class BaseTargetingEntity(BaseGameEntity):
    target: GameEntity | None = None
    targeting: TargetingSubState
    visible = True
    threat_level: int = 50
    _substates_manifest = (
        ("spawn", SpawnSubState),
        ("targeting", TargetingSubState),
    )
    
    def assess_target(self, target: GameEntity) -> None:

        super().update(target=target)
        

In [12]:
a = BaseTargetingEntity(store=GameStore())
a.visible = False
a.threat_level = 0
a.update()
print(a.targeting.state)


a.location = TileCoordinate.from_tuple((10,10), parent_map_size=TileTuple(([100],[100])))
a.update()
print(a.targeting.state, a.state_vector)

a.visible = True
a.update()
print(a.targeting.state, a.state_vector)

a.threat_level = 50
a.update()
print(a.targeting.state, a.state_vector)

a.visible = False
a.update()
print(a.targeting.state, a.state_vector)

unknown
searching {'on_map': True, 'target_in_fov': False, 'target_is_hostile': False}
tracking {'on_map': True, 'target_in_fov': True, 'target_is_hostile': False}
targeting {'on_map': True, 'target_in_fov': True, 'target_is_hostile': True}
searching {'on_map': True, 'target_in_fov': False, 'target_is_hostile': False}


In [17]:
class DamageSubState(SpawnSubState):
    serious_threshold: int = 40
    critical_threshold: int = 10
    
    _state_bits = ('has_no_damage', 'has_minor_damage', 'has_serious_damage', 'has_critical_damage', 'is_destroyed')
    _states = ({'name':'good', 'on_enter':['update']}, 
                         {'name':'fair', 'on_enter':['update']},
                         {'name':'serious', 'on_enter':['update']},
                         {'name':'critical', 'on_enter':['update']},
                         {'name':'destroyed'},)
    _transitions = (            
            {'trigger':'update', 'source':['good', 'serious', 'critical'], 'dest':'fair', 'conditions':['is_on_map', 'has_minor_damage']},
            {'trigger':'update', 'source':['good', 'fair', 'critical'], 'dest':'serious', 'conditions':['is_on_map', 'has_serious_damage']},
            {'trigger':'update', 'source':['good', 'fair', 'serious'], 'dest':'critical', 'conditions':['is_on_map', 'has_critical_damage']},
            {'trigger':'update', 'source':['fair', 'serious', 'critical'], 'dest':'good', 'conditions':['is_on_map', 'has_no_damage']},
            {'trigger':'update', 'source':['good', 'fair', 'serious', 'critical'], 'dest':'destroyed', 'conditions':['is_on_map', 'is_destroyed']},
            {'trigger':'update', 'source':'destroyed', 'dest':'destroyed'})
    _initial_state = 'good'
    
    def set_bits(self) -> None:
        self.store.state_vector['has_no_damage'] = self.store.hp == self.store.max_hp  # type: ignore
        self.store.state_vector['has_minor_damage'] = self.serious_threshold < self.store.hp <= self.store.max_hp - 1 # type: ignore
        self.store.state_vector['has_serious_damage'] = self.critical_threshold < self.store.hp <= self.serious_threshold  # type: ignore
        self.store.state_vector['has_critical_damage'] = 0 < self.store.hp <= self.critical_threshold  # type: ignore
        self.store.state_vector['is_destroyed'] = self.store.hp == 0  # type: ignore

    # Based on HP for now. Can be expanded later.
    def has_no_damage(self) -> bool: 
        return self.store.state_vector['has_no_damage'] # type: ignore
    
    def has_minor_damage(self) -> bool:
        return self.store.state_vector['has_minor_damage'] # type: ignore
    
    def has_serious_damage(self) -> bool:
        return self.store.state_vector['has_serious_damage'] # type: ignore
    
    def has_critical_damage(self) -> bool:
        return self.store.state_vector['has_critical_damage'] # type: ignore

    def is_destroyed(self) -> bool:
        return self.store.state_vector['is_destroyed'] # type: ignore


class PhysicalDamageEntity(BaseGameEntity):
    max_hp: int
    hp: int
    physical_condition: DamageSubState

    _substates_manifest = (
        ("spawn", SpawnSubState),
        ("physical_condition", DamageSubState),
    )

    def __init__(self,
                 store: GameStore,
                 *,
                 max_hp: int = 100,
                 hp: int = 100,
                 location: Tuple[int, int] | TileCoordinate | None = None,
                 name: str="<Unnamed>", 
                 symbol: str=' ', 
                 color: Tuple[int, int, int]=(0,0,0)) -> None:
        self.max_hp = max_hp
        self.hp = hp
        
        super().__init__(store=store, location=location, name=name, symbol=symbol, color=color)

    def take_damage(self, amount: int) -> None:
        if self.physical_condition.is_destroyed():
            return
        self.hp = max(0, self.hp - amount)
        self.update()
    
    def make_repair(self, amount: int) -> None:
        if self.physical_condition.is_destroyed():
            return
        self.hp = min(self.max_hp, self.hp + amount)
        self.update()



In [18]:
b = PhysicalDamageEntity(store=GameStore()) # type: ignore
b.location = TileCoordinate.from_tuple((10,10), parent_map_size=TileTuple(([100],[100])))
b.update()
print(b.state_vector, b.physical_condition.state, b.hp)
b.hp

{'on_map': True, 'has_no_damage': True, 'has_minor_damage': False, 'has_serious_damage': False, 'has_critical_damage': False, 'is_destroyed': False} good 100


100

In [20]:
a = PhysicalDamageEntity(store=GameStore())
a.location = TileCoordinate.from_tuple((10,10), parent_map_size=TileTuple(([100],[100])))
print(a.hp,a.max_hp) # Troubleshoot substate with dummy parent

print(a.hp,a.physical_condition.state, a.state_vector)
a.take_damage(95)
print(a.hp,a.physical_condition.state, a.state_vector)
a.make_repair(50)
print(a.hp,a.physical_condition.state, a.state_vector)
a.make_repair(40)
print(a.hp,a.physical_condition.state, a.state_vector)
a.take_damage(105)
print(a.hp,a.physical_condition.state, a.state_vector)
a.make_repair(200)
print(a.hp,a.physical_condition.state, a.state_vector)

100 100
100 good {'on_map': False, 'has_no_damage': True, 'has_minor_damage': False, 'has_serious_damage': False, 'has_critical_damage': False, 'is_destroyed': False}
5 critical {'on_map': True, 'has_no_damage': False, 'has_minor_damage': False, 'has_serious_damage': False, 'has_critical_damage': True, 'is_destroyed': False}
55 fair {'on_map': True, 'has_no_damage': False, 'has_minor_damage': True, 'has_serious_damage': False, 'has_critical_damage': False, 'is_destroyed': False}
95 fair {'on_map': True, 'has_no_damage': False, 'has_minor_damage': True, 'has_serious_damage': False, 'has_critical_damage': False, 'is_destroyed': False}
0 destroyed {'on_map': True, 'has_no_damage': False, 'has_minor_damage': False, 'has_serious_damage': False, 'has_critical_damage': False, 'is_destroyed': True}
0 destroyed {'on_map': True, 'has_no_damage': False, 'has_minor_damage': False, 'has_serious_damage': False, 'has_critical_damage': False, 'is_destroyed': True}


In [24]:
a.physical_condition.may_to_serious()

True

In [ ]:
a.hp

In [ ]:
class CombatSubState(BaseSubState):
    _has_target: bool = False
    _is_target: bool = False
    _distance_to_target: int = 9999
    
    def __init__(self, name: str, parent: StatefulObject) -> None:
        super().__init__(name=name, store=parent)

    def is_in_melee_range(self) -> bool:
        return self._distance_to_target <= 1
    
    def is_in_missile_range(self) -> bool:
        return 1 < self._distance_to_target <= 5
    
    def in_in_spell_range(self) -> bool:
        return 5 < self._distance_to_target <= 10
    
    def is_out_of_range(self) -> bool:
        return self._distance_to_target > 10
    
    def has_target(self) -> bool:
        return self._has_target

    def has_no_target(self) -> bool:
        return not self.has_target()
    
    def is_target(self) -> bool:
        return self._is_target
    

class BaseCombatEntity(BaseTargetableEntity, BaseTargetingEntity):
    machine: Machine
    combat: CombatSubState

    def __init__(self,
                 store: GameStore,
                 *,                 
                 location: Tuple[int, int] | TileCoordinate | None = None,
                 name: str="<Unnamed>", 
                 symbol: str=' ', 
                 color: Tuple[int, int, int]=(0,0,0)) -> None:
        
        super().__init__(store=store, location=location, symbol=symbol, color=color, name=name)
        
        combat = CombatSubState(name='combat_status', parent=self)
        combat_states = [{'name':'engaged', 'on_enter':['update']},
                         {'name': 'melee_ready', 'on_enter':['update']},
                         {'name': 'missile_ready', 'on_enter':['update']},
                         {'name': 'spell_ready', 'on_enter':['update']}, 
                         {'name':'disengaged', 'on_enter':['update']}]
        combat_transitions = [
            {'trigger':'update', 'source':['disengaged', 'melee_ready', 'missile_ready', 'spell_ready'], 'dest':'engaged', 'conditions':['is_activated', 'has_target', 'is_target']},
            {'trigger':'update', 'source':'*', 'dest':'melee_ready', 'conditions':['is_activated', 'has_target', 'is_target', 'is_in_melee_range']},
            {'trigger':'update', 'source':'*', 'dest':'missile_ready', 'conditions':['is_activated', 'has_target', 'is_target', 'is_in_missile_range']},
            {'trigger':'update', 'source':'*', 'dest':'spell_ready', 'conditions':['is_activated', 'has_target', 'is_target', 'in_in_spell_range']},
            {'trigger':'update', 'source':['melee_ready', 'missile_ready', 'spell_ready'], 'dest':'disengaged', 'conditions':['is_activated', 'has_no_target', 'is_out_of_range']},
            {'trigger':'engage', 'source':'disengaged', 'dest':'engaged', 'conditions':['is_activated', 'has_target']},
            {'trigger':'disengage', 'source':'engaged', 'dest':'disengaged', 'conditions':['is_activated', 'has_no_target', 'is_out_of_range']},
        ]
        combat_machine = Machine(model=combat, states=combat_states, transitions=combat_transitions, initial='disengaged')
        combat.machine = combat_machine
        self.combat = combat    
        self.substates.append(combat)
    
    def distance_to_target(self) -> int:

        if self.target and self.location and self.target.location is not None:
            dx = self.target.location.x - self.location.x
            dy = self.target.location.y - self.location.y
            return max(abs(dx), abs(dy))  # Using Chebyshev distance for grid-based movement
        
        return 9999
    
    def attack(self) -> int:
        if self.targeting.is_targeting(): #type: ignore
            pass

        raise NotImplementedError("Attack method must be implemented in subclasses.")
    
    def update(self) -> None:
        super().update()
        for substate in self.substates:
            if substate.name == 'combat_status': #type: ignore
                if self.target is not None:
                    distance = self.distance_to_target()
                    if distance is not None:
                        substate._distance_to_target = distance  # type: ignore

                

                substate.update() # type: ignore
                time.sleep(0.1)

In [ ]:
a = BaseCombatEntity(store=GameStore())
b = BaseCombatEntity(store=GameStore())
a.targeter = b
b.targeter = a
a.target = b
b.target = a
a.visible = True
a.hostile = True
b.visible = True
b.hostile = True
a.location = TileCoordinate.from_tuple((5,5), parent_map_size=TileTuple(([100],[100])))
b.location = TileCoordinate.from_tuple((7,7), parent_map_size=TileTuple(([100],[100])))
a.activate()

In [ ]:
print(a.combat.state)
a.update()
print(a.combat.state)

In [ ]:
a.distance_to_target()